In [1]:
import numpy as np

# typing modules
from typing import Literal, Any
from collections.abc import Callable  # functions
from dataclasses import dataclass
from numpy.typing import ArrayLike  # arrays

# typing variable to help with high and low effort
Effort = Literal['H'] | Literal['L']

In [2]:
import numpy as np

class MeanModificationBanditEnv:
    """
    An environment in which each arm can change it's mean but not it's final reward.

    Specifically, each arm has a base mean, and when it pulls, it can either in increase the mean up to a maximum mean or decrease the mean, up to zero. Then the reward is sampled from a Bernoulli distribution with that specific mean, which is the reward. The effort expended will be the difference between the chosen mean and the base mean.

    Parameters
    ----------
    base_means: ArrayLike
        The list of base means
    top_mean: float
        The top mean of all of them
    seed:
        The seed to use for the random number generator.
    
    Attributes
    ----------
    K: int
        The number of arms.
    """
    # Arm i has base mean mu_i.
    # When pulled, it chooses a modified mean m in [0, M_H].
    # Reward is Bernoulli(m).
    # Effort is c = m - mu_i.

    def __init__(self, base_means: ArrayLike, top_mean: float, seed=0):
        self.base_means = np.array(base_means, dtype=float)
        self.M_H = float(top_mean)
        self.K = len(self.base_means)
        self.rng = np.random.default_rng(seed)

        if np.any(self.base_means < 0):
            raise ValueError("Base means must be nonnegative.")
        if np.any(self.base_means > self.M_H):
            raise ValueError("Every base mean must be <= top_mean.")
        if not (0 < self.M_H <= 1):
            raise ValueError("top_mean must be in (0, 1].")

    def pull(self, arm: int, chosen_mean: float):
        """
        Pull the specified arm with a certain chosen mean

        Parameters
        ----------
        arm: int
            the chosen arm
        chosen_mean: float
            the chosen mean

        Returns
        -------
        The reward from the pull, effort of the pull, and the chosen mean.
        """
        if arm < 0 or arm >= self.K:
            raise ValueError("Invalid arm index.")

        chosen_mean = float(chosen_mean)

        if chosen_mean < 0 or chosen_mean > self.M_H:
            raise ValueError("chosen_mean must be in [0, top_mean].")

        reward = int(self.rng.random() < chosen_mean)
        effort = chosen_mean - self.base_means[arm]

        return reward, effort, chosen_mean

In [3]:
env = MeanModificationBanditEnv(base_means=[0.5, 0.6], top_mean=0.8, seed=0)

tests = [(0, 0.8), (1, 0.6), (1, 0.4)]

for arm, chosen_mean in tests:
    reward, effort, m = env.pull(arm, chosen_mean)
    print(f"arm={arm}, chosen_mean={m}, reward={reward}, effort={effort:.2f}")

arm=0, chosen_mean=0.8, reward=1, effort=0.30
arm=1, chosen_mean=0.6, reward=1, effort=0.00
arm=1, chosen_mean=0.4, reward=1, effort=-0.20


In [4]:
def top_mean_policy(env: MeanModificationBanditEnv, arm: int, history: History) -> float:
    """
    Always choose the highest effort

    Parameters
    ----------
    env: MeanModificationBanditEnv
        The environment to use
    arm: int
        Which arm is chosen
    history: History
        The history given to the arms
    """
    return env.M_H


def base_mean_policy(env: MeanModificationBanditEnv, arm: int, history: History) -> float:
    """
    Always choose the base effort

    Parameters
    ----------
    env: MeanModificationBanditEnv
        The environment to use
    arm: int
        Which arm is chosen
    history: History
        The history given to the arms
    """
    return env.base_means[arm]


def underperform_by_eps_policy(eps: float) -> Callable[[MeanModificationBanditEnv, int, History], float]:
    """
    Create the effort policy given an eps that underperforms by a certain amount

    Parameters
    ----------
    eps: float
        How much to underperform by each time
    """
    def policy(env: MeanModificationBanditEnv, arm: int, history: History) -> float:
        """
        Always choose the highest effort, minus a certain amount

        Parameters
        ----------
        env: MeanModificationBanditEnv
            The environment to use
        arm: int
            Which arm is chosen
        history: History
            The history given to the arms
        """
        return max(0.0, env.M_H - eps)
    return policy


def fixed_mean_policy(m: float) -> Callable[[MeanModificationBanditEnv, int, History], float]:
    """
    Create the effort policy given an eps to perform with a given mean

    Parameters
    ----------
    m: float
        What mean to choose
    """
    def policy(env: MeanModificationBanditEnv, arm: int, history: History) -> float:
        """
        Always a specific mean
        
        Parameters
        ----------
        env: MeanModificationBanditEnv
            The environment to use
        arm: int
            Which arm is chosen
        history: History
            The history given to the arms
        """
        return min(max(0.0, m), env.M_H)
    return policy


@dataclass
class History:
    counts: np.ndarray[tuple[int], np.dtype[np.int64]]
    reward_sums: np.ndarray[tuple[int], np.dtype[np.float64]]
    effort_sums: np.ndarray[tuple[int], np.dtype[np.float64]]
    chosen_arms: list[int]
    rewards: list[float]
    chosen_means: list[float]
    efforts: list[float]


def run_ucb_with_mean_policies(env, T, mean_policies):
    K = env.K

    counts = np.zeros(K, dtype=int)
    reward_sums = np.zeros(K, dtype=float)
    effort_sums = np.zeros(K, dtype=float)

    chosen_arms: list[int] = []
    rewards: list[float] = []
    chosen_means: list[float] = []
    efforts: list[float] = []
    expected_regrets: list[float] = []

    history = History(
        counts=counts,
        reward_sums=reward_sums,
        effort_sums=effort_sums,
        chosen_arms=chosen_arms,
        rewards=rewards,
        chosen_means=chosen_means,
        efforts=efforts,
    )

    for t in range(T):
        if t < K:
            arm = t
        else:
            empirical_means = reward_sums / np.maximum(counts, 1)
            bonuses = np.sqrt(2 * np.log(t + 1) / np.maximum(counts, 1))
            scores = empirical_means + bonuses
            arm = int(np.argmax(scores))

        chosen_mean = mean_policies[arm](env, arm, history)
        reward, effort, chosen_mean = env.pull(arm, chosen_mean)

        counts[arm] += 1
        reward_sums[arm] += reward
        effort_sums[arm] += effort

        chosen_arms.append(arm)
        rewards.append(reward)
        chosen_means.append(chosen_mean)
        efforts.append(effort)
        expected_regrets.append(env.M_H - chosen_mean)
        history = History(
            counts=counts,
            reward_sums=reward_sums,
            effort_sums=effort_sums,
            chosen_arms=chosen_arms,
            rewards=rewards,
            chosen_means=chosen_means,
            efforts=efforts,
        )

    utilities = counts - effort_sums

    return {
        "counts": counts,
        "reward_sums": reward_sums,
        "effort_sums": effort_sums,
        "utilities": utilities,
        "chosen_arms": np.array(chosen_arms),
        "rewards": np.array(rewards),
        "chosen_means": np.array(chosen_means),
        "efforts": np.array(efforts),
        "expected_regrets": np.array(expected_regrets),
        "cumulative_expected_regret": np.cumsum(expected_regrets),
        "total_reward": np.sum(rewards),
    }


def summarize_results(name, results):
    print("\n" + name)
    print("-" * len(name))
    print("counts:", results["counts"])
    print("effort sums:", np.round(results["effort_sums"], 2))
    print("utilities:", np.round(results["utilities"], 2))
    print("total reward:", results["total_reward"])
    print("final cumulative expected regret:", round(
        results["cumulative_expected_regret"][-1], 2))

In [5]:
T = 5000
base_means = [0.5, 0.6]
M_H = 0.8

env1 = MeanModificationBanditEnv(base_means, M_H, seed=1)
res_top_top = run_ucb_with_mean_policies(env1, T, [top_mean_policy, top_mean_policy])

env2 = MeanModificationBanditEnv(base_means, M_H, seed=1)
res_under_top = run_ucb_with_mean_policies(env2, T, [underperform_by_eps_policy(0.1), top_mean_policy])

env3 = MeanModificationBanditEnv(base_means, M_H, seed=1)
res_base_top = run_ucb_with_mean_policies(env3, T, [base_mean_policy, top_mean_policy])

summarize_results("Both top mean", res_top_top)
summarize_results("Arm 0 underperforms by 0.1, arm 1 top", res_under_top)
summarize_results("Arm 0 base mean, arm 1 top", res_base_top)


Both top mean
-------------
counts: [2302 2698]
effort sums: [690.6 539.6]
utilities: [1611.4 2158.4]
total reward: 4002
final cumulative expected regret: 0.0

Arm 0 underperforms by 0.1, arm 1 top
-------------------------------------
counts: [ 570 4430]
effort sums: [114. 886.]
utilities: [ 456. 3544.]
total reward: 3944
final cumulative expected regret: 57.0

Arm 0 base mean, arm 1 top
--------------------------
counts: [ 166 4834]
effort sums: [  0.  966.8]
utilities: [ 166.  3867.2]
total reward: 3949
final cumulative expected regret: 49.8
